# Variable-star cross-match for a coordinate target list — v8

Previous name `lantern_variable_star_crossmatch_v8`

Input is a single target table with three columns:

```text
name,ra,dec
```

where `ra` and `dec` are in degrees (ICRS). Each row is treated as one target and is checked directly at its supplied sky position.

For each target, the notebook checks:

1. AAVSO VSX;
2. OGLE (three catalog families).

Gaia is intentionally **not queried** in this version because the Gaia Archive can be slow or unstable.

There is **no** Second-Pass candidate parquet, `target_id`, member-locus mapping, ANTARES-locus lookup, or Gaia query.


## Input format

By default the notebook reads:

```text
targets.csv
```

Example:

```csv
name,ra,dec
object_001,150.12345,2.34567
object_002,151.54321,1.23456
```

`name` must be non-empty and unique. `ra` and `dec` must be numeric degrees.


## Dependencies

This notebook uses:

```text
pandas
numpy
astropy
astroquery
```

If `astroquery` is missing:

```bash
pip install astroquery
```


In [ ]:
from pathlib import Path
import json
import time
from collections.abc import Mapping

import numpy as np
import pandas as pd

import astropy.units as u
from astropy.coordinates import SkyCoord

from astroquery.vizier import Vizier


# ============================================================
# INPUT
# ============================================================

TARGET_FILE = Path("targets.csv")


# ============================================================
# OUTPUT
# ============================================================

OUTPUT_DIR = Path("variable_star_crossmatch")

TARGET_SUMMARY_FILE = (
    OUTPUT_DIR
    / "target_variable_star_summary.csv"
)

MATCHES_FILE = (
    OUTPUT_DIR
    / "external_variable_catalog_matches.csv"
)


# ============================================================
# QUERY SETTINGS
# ============================================================

MATCH_RADIUS_ARCSEC = 2.0

RUN_VSX = True
RUN_OGLE = True

# Small pause between remote catalog requests.
NETWORK_PAUSE_SEC = 0.15

# Set to None to disable partial saves.
SAVE_EVERY_N_TARGETS = 10


# ============================================================
# CATALOGS
# ============================================================

VSX_CATALOG = "B/vsx/vsx"

OGLE_VIZIER_CATALOGS = {
    "J/AcA/69/321": "OGLE RR Lyrae - Galactic bulge/disk",
    "J/AcA/66/131": "OGLE RR Lyrae - Magellanic System",
    "J/AcA/68/315": "OGLE Galactic Cepheids",
}


In [ ]:
# ============================================================
# CLASSIFICATION LABELS
# ============================================================

NONSTELLAR_TOKENS = (
    "AGN",
    "QSO",
    "QUASAR",
    "BLAZAR",
    "BLLAC",
    "BL LAC",
    "SUPERNOVA",
)

VSX_FLAG_MEANING = {
    0: "variable",
    1: "suspected variable",
    2: "constant or non-existent",
    3: "possible duplicate",
}


In [ ]:
# ============================================================
# GENERIC HELPERS
# ============================================================

def clean_scalar(value):
    """Convert scalar masked/Numpy values to plain Python values."""
    if np.ma.is_masked(value):
        return None

    if isinstance(value, bytes):
        return value.decode(
            "utf-8",
            errors="replace",
        )

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        (
            Mapping,
            list,
            tuple,
            set,
            np.ndarray,
        ),
    ):
        return value

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def json_safe(value):
    """Recursively convert values to JSON-serializable Python objects."""
    value = clean_scalar(value)

    if isinstance(value, Mapping):
        return {
            str(k): json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, np.ndarray):
        return [
            json_safe(v)
            for v in value.tolist()
        ]

    if isinstance(value, (list, tuple, set)):
        return [
            json_safe(v)
            for v in value
        ]

    return value


def json_text(value):
    return json.dumps(
        json_safe(value),
        ensure_ascii=False,
        sort_keys=True,
    )


def row_to_dict(row):
    """Convert an Astropy Row to a plain dictionary robustly."""
    names = getattr(row, "colnames", None)

    if names is None:
        table = getattr(row, "table", None)
        if table is not None:
            names = getattr(table, "colnames", None)

    if names is None:
        try:
            names = list(row.keys())
        except Exception as exc:
            raise TypeError(
                "Could not determine Astropy row column names."
            ) from exc

    return {
        str(name): json_safe(row[name])
        for name in names
    }


def first_present(mapping, names):
    for name in names:
        if name in mapping:
            value = clean_scalar(mapping[name])
            if value is not None and str(value).strip() != "":
                return value
    return None


def unique_text(values):
    seen = set()
    output = []

    for value in values:
        value = clean_scalar(value)
        if value is None:
            continue

        text = str(value).strip()
        if not text or text.lower() == "nan":
            continue

        if text not in seen:
            seen.add(text)
            output.append(text)

    return ";".join(output)


def to_float(value):
    value = clean_scalar(value)

    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def is_nonstellar_label(value):
    value = clean_scalar(value)

    if value is None:
        return False

    text = str(value).strip().upper()
    return any(token in text for token in NONSTELLAR_TOKENS)


def angular_sep_arcsec(ra1, dec1, ra2, dec2):
    c1 = SkyCoord(
        ra=float(ra1) * u.deg,
        dec=float(dec1) * u.deg,
        frame="icrs",
    )
    c2 = SkyCoord(
        ra=float(ra2) * u.deg,
        dec=float(dec2) * u.deg,
        frame="icrs",
    )
    return float(c1.separation(c2).arcsec)


In [ ]:
# ============================================================
# LOAD AND VALIDATE TARGET LIST
# ============================================================

targets = pd.read_csv(TARGET_FILE)

required_columns = {"name", "ra", "dec"}
missing = required_columns - set(targets.columns)

if missing:
    raise KeyError(
        "Target file is missing required columns: "
        f"{sorted(missing)}"
    )

targets = targets[["name", "ra", "dec"]].copy()

targets["name"] = targets["name"].astype(str).str.strip()
targets["ra"] = pd.to_numeric(targets["ra"], errors="coerce")
targets["dec"] = pd.to_numeric(targets["dec"], errors="coerce")

if targets.empty:
    raise ValueError(f"No targets found in {TARGET_FILE}")

if targets["name"].eq("").any():
    raise ValueError("Some target names are empty.")

if targets["name"].duplicated().any():
    duplicated = targets.loc[
        targets["name"].duplicated(keep=False),
        "name",
    ].tolist()
    raise ValueError(
        "Target names must be unique. Duplicates: "
        f"{duplicated}"
    )

if targets[["ra", "dec"]].isna().any().any():
    bad = targets.loc[
        targets[["ra", "dec"]].isna().any(axis=1)
    ]
    raise ValueError(
        "Some RA/Dec values are not numeric:\n"
        + bad.to_string(index=False)
    )

if ((targets["ra"] < 0) | (targets["ra"] >= 360)).any():
    bad = targets.loc[
        (targets["ra"] < 0) | (targets["ra"] >= 360)
    ]
    raise ValueError(
        "RA must satisfy 0 <= ra < 360 deg:\n"
        + bad.to_string(index=False)
    )

if ((targets["dec"] < -90) | (targets["dec"] > 90)).any():
    bad = targets.loc[
        (targets["dec"] < -90) | (targets["dec"] > 90)
    ]
    raise ValueError(
        "Dec must satisfy -90 <= dec <= 90 deg:\n"
        + bad.to_string(index=False)
    )

targets = targets.reset_index(drop=True)

print(f"Loaded {len(targets):,} targets from {TARGET_FILE}")
display(targets.head())


In [ ]:
# ============================================================
# AAVSO VSX
# ============================================================

def query_vsx(
    ra,
    dec,
    radius_arcsec=MATCH_RADIUS_ARCSEC,
):
    """Query AAVSO VSX through VizieR."""

    coord = SkyCoord(
        ra=float(ra) * u.deg,
        dec=float(dec) * u.deg,
        frame="icrs",
    )

    vizier = Vizier(
        columns=[
            "Name",
            "V",
            "RAJ2000",
            "DEJ2000",
            "Type",
            "max",
            "min",
            "Period",
            "Sp",
        ],
        row_limit=-1,
    )

    tables = vizier.query_region(
        coord,
        radius=float(
            radius_arcsec
        ) * u.arcsec,
        catalog=VSX_CATALOG,
    )

    matches = []

    for table in tables:
        for row in table:
            row_dict = row_to_dict(
                row
            )

            source_ra = to_float(
                row_dict.get(
                    "RAJ2000"
                )
            )

            source_dec = to_float(
                row_dict.get(
                    "DEJ2000"
                )
            )

            if (
                np.isfinite(
                    source_ra
                )
                and np.isfinite(
                    source_dec
                )
            ):
                sep = angular_sep_arcsec(
                    ra,
                    dec,
                    source_ra,
                    source_dec,
                )
            else:
                sep = np.nan

            max_mag = to_float(
                row_dict.get(
                    "max"
                )
            )

            min_mag = to_float(
                row_dict.get(
                    "min"
                )
            )

            mag_values = [
                value
                for value
                in (
                    max_mag,
                    min_mag,
                )
                if np.isfinite(
                    value
                )
            ]

            representative_mag = (
                float(
                    np.mean(
                        mag_values
                    )
                )
                if mag_values
                else np.nan
            )

            vsx_flag = clean_scalar(
                row_dict.get(
                    "V"
                )
            )

            try:
                vsx_flag_int = int(
                    vsx_flag
                )
            except (
                TypeError,
                ValueError,
            ):
                vsx_flag_int = None

            matches.append(
                {
                    "service": "VSX",
                    "catalog": VSX_CATALOG,
                    "source_name": clean_scalar(
                        row_dict.get(
                            "Name"
                        )
                    ),
                    "source_ra": source_ra,
                    "source_dec": source_dec,
                    "separation_arcsec": sep,
                    "class_name": clean_scalar(
                        row_dict.get(
                            "Type"
                        )
                    ),
                    "class_score": np.nan,
                    "magnitude": representative_mag,
                    "period_days": to_float(
                        row_dict.get(
                            "Period"
                        )
                    ),
                    "variability_flag": vsx_flag_int,
                    "raw_json": json_text(
                        row_dict
                    ),
                }
            )

    matches.sort(
        key=lambda x: (
            np.inf
            if not np.isfinite(
                x[
                    "separation_arcsec"
                ]
            )
            else x[
                "separation_arcsec"
            ]
        )
    )

    return matches


In [ ]:
# ============================================================
# OGLE THROUGH VIZIER
# ============================================================

def query_ogle(
    ra,
    dec,
    radius_arcsec=MATCH_RADIUS_ARCSEC,
):
    """
    Cross-match a curated set of major OGLE variable-star catalogs.

    All catalog families are sent in one VizieR region query to avoid
    one network request per catalog.  Different OGLE tables use different
    column names, so complete rows are retained as JSON and common fields
    are extracted when possible.
    """

    coord = SkyCoord(
        ra=float(ra) * u.deg,
        dec=float(dec) * u.deg,
        frame="icrs",
    )

    vizier = Vizier(
        columns=[
            "**",
            "_RAJ2000",
            "_DEJ2000",
        ],
        row_limit=-1,
    )

    tables = vizier.query_region(
        coord,
        radius=float(
            radius_arcsec
        ) * u.arcsec,
        catalog=list(
            OGLE_VIZIER_CATALOGS.keys()
        ),
    )

    matches = []

    for table_name in tables.keys():
        table = tables[
            table_name
        ]

        table_name_text = str(
            table_name
        )

        catalog_family = next(
            (
                catalog_id
                for catalog_id
                in OGLE_VIZIER_CATALOGS
                if catalog_id
                in table_name_text
            ),
            table_name_text,
        )

        catalog_label = (
            OGLE_VIZIER_CATALOGS.get(
                catalog_family,
                table_name_text,
            )
        )

        for row in table:
            row_dict = row_to_dict(
                row
            )

            source_name = first_present(
                row_dict,
                [
                    "Name",
                    "OGLE",
                    "Star",
                    "ID",
                    "Object",
                ],
            )

            class_name = first_present(
                row_dict,
                [
                    "Type",
                    "Subtype",
                    "RR",
                    "Class",
                    "VarType",
                ],
            )

            if class_name is None:
                class_name = catalog_label

            source_ra = to_float(
                first_present(
                    row_dict,
                    [
                        "_RAJ2000",
                        "RAJ2000",
                        "RAdeg",
                        "RA_ICRS",
                        "RA",
                    ],
                )
            )

            source_dec = to_float(
                first_present(
                    row_dict,
                    [
                        "_DEJ2000",
                        "DEJ2000",
                        "DEdeg",
                        "DE_ICRS",
                        "DEC",
                        "Dec",
                    ],
                )
            )

            if (
                np.isfinite(
                    source_ra
                )
                and np.isfinite(
                    source_dec
                )
            ):
                sep = angular_sep_arcsec(
                    ra,
                    dec,
                    source_ra,
                    source_dec,
                )
            else:
                # VizieR query_region already enforces the search radius.
                sep = np.nan

            magnitude = first_present(
                row_dict,
                [
                    "<Imag>",
                    "Imag",
                    "I",
                    "<Vmag>",
                    "Vmag",
                    "V",
                ],
            )

            period = first_present(
                row_dict,
                [
                    "Per",
                    "Period",
                    "P",
                    "P_1",
                ],
            )

            matches.append(
                {
                    "service": "OGLE",
                    "catalog": table_name_text,
                    "catalog_family": catalog_family,
                    "catalog_description": catalog_label,
                    "source_name": source_name,
                    "source_ra": source_ra,
                    "source_dec": source_dec,
                    "separation_arcsec": sep,
                    "class_name": class_name,
                    "class_score": np.nan,
                    "magnitude": to_float(
                        magnitude
                    ),
                    "period_days": to_float(
                        period
                    ),
                    "variability_flag": None,
                    "raw_json": json_text(
                        row_dict
                    ),
                }
            )

    # Remove repeated representations of the same source/class.
    # If no usable source identifier exists, use the raw row itself
    # instead of collapsing all anonymous rows together.
    deduped = []
    seen = set()

    for row in matches:
        source_key = row.get(
            "source_name"
        )

        if source_key is None:
            source_key = row.get(
                "raw_json"
            )

        key = (
            str(
                row.get(
                    "catalog_family"
                )
            ),
            str(
                source_key
            ),
            str(
                row.get(
                    "class_name"
                )
            ),
        )

        if key in seen:
            continue

        seen.add(
            key
        )

        deduped.append(
            row
        )

    deduped.sort(
        key=lambda x: (
            np.inf
            if not np.isfinite(
                x[
                    "separation_arcsec"
                ]
            )
            else x[
                "separation_arcsec"
            ]
        )
    )

    return deduped


In [ ]:
# ============================================================
# EVIDENCE LOGIC
# ============================================================

def vsx_is_variable_candidate(row):
    return row.get("variability_flag") in {0, 1}


def match_is_stellar_variable(row):
    service = row.get("service")
    class_name = row.get("class_name")

    if service == "VSX":
        return (
            vsx_is_variable_candidate(row)
            and not is_nonstellar_label(class_name)
        )

    if service == "OGLE":
        # Every catalog in OGLE_VIZIER_CATALOGS is a curated
        # stellar-variable / stellar-binary catalog.
        return True

    return False


def match_is_agn_like(row):
    return is_nonstellar_label(row.get("class_name"))


def make_evidence_text(external_matches):
    stellar = []
    agn = []

    for row in external_matches:
        service = row.get("service")
        source = row.get("source_name")
        class_name = row.get("class_name")
        sep = to_float(row.get("separation_arcsec"))

        if np.isfinite(sep):
            sep_text = f"{sep:.2f}arcsec"
        else:
            sep_text = "within_radius"

        detail = (
            f"{service}:{class_name}"
            f"[{source};{sep_text}]"
        )

        if match_is_stellar_variable(row):
            stellar.append(detail)

        if match_is_agn_like(row):
            agn.append(detail)

    return unique_text(stellar), unique_text(agn)


In [ ]:
# ============================================================
# MAIN LOOP
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

target_rows = []
match_rows = []


def save_partial():
    if target_rows:
        pd.DataFrame(target_rows).to_csv(
            OUTPUT_DIR / "_partial_target_summary.csv",
            index=False,
        )

    if match_rows:
        pd.DataFrame(match_rows).to_csv(
            OUTPUT_DIR / "_partial_external_matches.csv",
            index=False,
        )


for target_number, target in enumerate(
    targets.itertuples(index=False),
    start=1,
):
    name = str(target.name)
    ra = float(target.ra)
    dec = float(target.dec)

    print(
        f"[{target_number}/{len(targets)}] "
        f"{name}  ra={ra:.6f}  dec={dec:.6f}"
    )

    errors = []
    external_matches = []

    # --------------------------------------------------------
    # VSX
    # --------------------------------------------------------

    if RUN_VSX:
        try:
            external_matches.extend(
                query_vsx(ra, dec)
            )
        except Exception as exc:
            errors.append(
                "VSX: "
                f"{type(exc).__name__}: {exc}"
            )

        if NETWORK_PAUSE_SEC:
            time.sleep(NETWORK_PAUSE_SEC)

    # --------------------------------------------------------
    # OGLE
    # --------------------------------------------------------

    if RUN_OGLE:
        try:
            external_matches.extend(
                query_ogle(ra, dec)
            )
        except Exception as exc:
            errors.append(
                "OGLE: "
                f"{type(exc).__name__}: {exc}"
            )

    # --------------------------------------------------------
    # Detailed match rows
    # --------------------------------------------------------

    for match in external_matches:
        row = {
            "name": name,
            "target_ra": ra,
            "target_dec": dec,
        }
        row.update(match)
        match_rows.append(row)

    # --------------------------------------------------------
    # Per-target summary
    # --------------------------------------------------------

    vsx_rows = [
        row for row in external_matches
        if row.get("service") == "VSX"
    ]
    ogle_rows = [
        row for row in external_matches
        if row.get("service") == "OGLE"
    ]

    stellar_evidence, agn_evidence = make_evidence_text(
        external_matches
    )

    target_rows.append(
        {
            "name": name,
            "ra": ra,
            "dec": dec,
            "vsx_n_matches": len(vsx_rows),
            "vsx_names": unique_text(
                row.get("source_name") for row in vsx_rows
            ),
            "vsx_types": unique_text(
                row.get("class_name") for row in vsx_rows
            ),
            "ogle_n_matches": len(ogle_rows),
            "ogle_names": unique_text(
                row.get("source_name") for row in ogle_rows
            ),
            "ogle_types": unique_text(
                row.get("class_name") for row in ogle_rows
            ),
            "ogle_catalogs": unique_text(
                row.get(
                    "catalog_family",
                    row.get("catalog"),
                )
                for row in ogle_rows
            ),
            "likely_variable_star": bool(stellar_evidence),
            "variable_star_evidence": stellar_evidence,
            "agn_evidence": agn_evidence,
            "query_errors": unique_text(errors),
        }
    )

    if (
        SAVE_EVERY_N_TARGETS
        and target_number % SAVE_EVERY_N_TARGETS == 0
    ):
        save_partial()


target_summary = pd.DataFrame(target_rows)
external_matches = pd.DataFrame(match_rows)

print()
print(f"Target-summary rows: {len(target_summary):,}")
print(f"External-match rows: {len(external_matches):,}")

display(target_summary)


In [ ]:
# ============================================================
# SAVE FINAL OUTPUTS
# ============================================================

target_summary.to_csv(
    TARGET_SUMMARY_FILE,
    index=False,
)

external_matches.to_csv(
    MATCHES_FILE,
    index=False,
)

print("Saved:")
print(f"  {TARGET_SUMMARY_FILE}")
print(f"  {MATCHES_FILE}")


## Interpreting the output

Main file:

```text
variable_star_crossmatch/target_variable_star_summary.csv
```

Important columns:

- `name`, `ra`, `dec`: copied from the input target list.
- `likely_variable_star`: at least one successful VSX or OGLE match provides stellar-variable evidence within the adopted match radius.
- `variable_star_evidence`: source/classification details that triggered the flag.
- `agn_evidence`: AGN/QSO-like labels encountered in the queried catalogs, kept separate from stellar-variable evidence.
- `vsx_n_matches`, `vsx_names`, `vsx_types`: AAVSO VSX results.
- `ogle_n_matches`, `ogle_names`, `ogle_types`, `ogle_catalogs`: results from the curated OGLE catalog list.
- `query_errors`: any VSX or OGLE remote-service failures.

There are **no Gaia queries or Gaia output columns in v8**.

For auditing individual catalog matches:

```text
variable_star_crossmatch/external_variable_catalog_matches.csv
```

Each row in that file includes the input target name/position, matched catalog source, angular separation, class, and raw catalog row.
